# Tamil sentiment: tamil-sentiment-distilbert

Using this [model](https://huggingface.co/Vasanth/tamil-sentiment-distilbert) and this [dataset](https://huggingface.co/datasets/community-datasets/tamilmixsentiment), both from HuggingFace (s/o Michael for finding these!).

## Setup

In [1]:
import torch

# setting up local server to use my laptop's GPU (doc here at https://docs.pytorch.org/docs/stable/notes/mps.html)
print('MPS available:', torch.backends.mps.is_available())
print('MPS built:', torch.backends.mps.is_built())

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('Using device:', device)

MPS available: False
MPS built: False
Using device: cpu


In [ ]:
# no peft/bitsandbytes/accelerate needed since tamil model is just a distilbert classifier
!pip install -q -U transformers torch pandas scikit-learn matplotlib tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/423.1 MB 22.3 MB/s eta 0:00:14

In [ ]:
import os
import pandas as pd

DATASET_ROOT = '/Users/michelle/Library/CloudStorage/GoogleDrive-xsmichelletk@gmail.com/.shortcut-targets-by-id/1BGQNTF4wTH0-VPksYIDj3o34N8lnu8WD/IAT 360 Final Project/Datasets'
TEST_DIR = os.path.join(DATASET_ROOT, 'Test')

print('Test folder contents:', os.listdir(TEST_DIR))

## Load + clean the Tamil test set

No train/test split here, same call Laraine and Michael made for Korean/Hinglish: the model is already trained, we're only evaluating it, so the whole `Test/df_tamil_cleaned.csv` file is the eval set. (Aaron's English RoBERTa run needed a stratified split because he was carving a test slice out of the shared 1M-row base dataset, that doesn't apply here.)

In [ ]:
TAMIL_TEST_CSV = os.path.join(TEST_DIR, 'df_tamil_cleaned.csv')
df_tamil_test = pd.read_csv(TAMIL_TEST_CSV)

# cleaning up duplicate cells in the csv file instead of manually going in
dupes = df_tamil_test.duplicated().sum()
df_tamil_test = df_tamil_test.drop_duplicates().reset_index(drop=True)
print(f'Dropped {dupes} duplicate rows')

print(df_tamil_test.shape)
print(df_tamil_test['Sentiment'].value_counts())
df_tamil_test.head()

## Load the Tamil model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# loading straight from the model card (https://huggingface.co/Vasanth/tamil-sentiment-distilbert)
MODEL_ID = 'Vasanth/tamil-sentiment-distilbert'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(device)
model.eval()

print('Model id2label:', model.config.id2label)

In [ ]:
# fixing the label map from the model card so it's consistent with the other languages
LABEL_MAP = {
    'LABEL_0': 'Positive',
    'LABEL_1': 'Negative',
    'LABEL_2': 'Unknown',  # Mixed_feelings
    'LABEL_3': 'Unknown',  # unknown_state
    'LABEL_4': 'Unknown',  # not-Tamil
}


def predict_sentiment(comment: str, max_length: int = 512) -> str:
    inputs = tokenizer(
        str(comment), return_tensors='pt', truncation=True, max_length=max_length
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    pred_id = logits.argmax(dim=-1).item()
    raw_label = model.config.id2label[pred_id]

    return LABEL_MAP.get(raw_label, 'Unknown')

## Classify

In [ ]:
from tqdm.auto import tqdm
import time

# running the full tamil set instead of subsampling since it's a light/fast model
SAMPLE_SIZE = None  # None = run on all rows
CHECKPOINT_EVERY = 100  # save progress every N comments
CHECKPOINT_PATH = os.path.join(TEST_DIR, 'df_tamil_distilbert_predictions_checkpoint.csv')

# build the eval set
eval_df = (
    df_tamil_test.sample(n=SAMPLE_SIZE, random_state=42) if SAMPLE_SIZE else df_tamil_test
).reset_index(drop=True)

# resume from checkpoint (if it exists)
if os.path.exists(CHECKPOINT_PATH):
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH)
    print(f"Found checkpoint with {len(checkpoint_df)} predictions already done. Resuming...")
    eval_df['Predicted_Sentiment'] = checkpoint_df['Predicted_Sentiment'].reindex(eval_df.index)
else:
    eval_df['Predicted_Sentiment'] = pd.NA
    print("No checkpoint found. Starting fresh.")

remaining_idx = eval_df.index[eval_df['Predicted_Sentiment'].isna()]
print(f"{len(remaining_idx)} comments left to classify out of {len(eval_df)} total.")

start_time = time.time()
for count, idx in enumerate(tqdm(remaining_idx, desc='Classifying with tamil-sentiment-distilbert')):
    comment = eval_df.loc[idx, 'CommentText']
    eval_df.loc[idx, 'Predicted_Sentiment'] = predict_sentiment(comment)

    # save a checkpoint periodically so a disconnect doesn't lose all progress :c
    if (count + 1) % CHECKPOINT_EVERY == 0:
        eval_df.to_csv(CHECKPOINT_PATH, index=False)

# final save
eval_df.to_csv(CHECKPOINT_PATH, index=False)
elapsed = time.time() - start_time
print(f"Done. Classified {len(remaining_idx)} comments in {elapsed/60:.1f} minutes.")
eval_df.head()

In [ ]:
# save predictions back to our shared Drive so the rest of the group can reuse them
RESULTS_PATH = os.path.join(TEST_DIR, 'df_tamil_distilbert_predictions.csv')
eval_df.to_csv(RESULTS_PATH, index=False)
print(f'Saved predictions to {RESULTS_PATH}')

## Confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# added Unknown to the label order since this model can output Mixed_feelings/unknown_state/not-Tamil, which don't map to our 3 classes
LABEL_ORDER = ['Positive', 'Negative', 'Neutral', 'Unknown']
DISPLAY_LABELS = ['Tamil - Positive', 'Tamil - Negative', 'Tamil - Neutral', 'Tamil - Unknown']

y_true = eval_df['Sentiment']
y_pred = eval_df['Predicted_Sentiment']

cm = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=DISPLAY_LABELS)
disp.plot(ax=ax, cmap='Blues', colorbar=True, xticks_rotation=30)
ax.set_title('tamil-sentiment-distilbert vs. Human Labels')
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, labels=LABEL_ORDER, target_names=DISPLAY_LABELS))